Mechanism Hypotheses:
1. Lateral exchange w/ surrounding water. For a cyclone moving south, water with low CHL enters and dilutes/reduces CHL concentration. 
2. Eddy-wind interaction. Ekman pumping drives upwelling in anticyclones and downwelling in cyclones. But if this is the case, then we should expect a change/response at the centers first.
3. Seasonal changes and mixed layer depth changes. 

In [ ]:
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from matplotlib.transforms import blended_transform_factory
from scipy.cluster.hierarchy import cophenet, dendrogram, fcluster, set_link_color_palette

from eddy_tracking.config import PROJECT_ROOT, load_analysis_config, read_settings, resolve_gold_dir, resolve_output_dir
from eddy_tracking.preprocess.streamline import compute_signed_distance_grid_km, index_centerlines_by_date
from eddy_tracking.preprocess.tracks import load_date_eddy_index
from eddy_tracking.utils.analysis_period import restrict_to_analysis_period
from eddy_tracking.utils.composite_regions import average_regions, list_composites
from eddy_tracking.utils.pigment_clusters import CLUSTER_CUT_HEIGHT, RATIO_PIGMENTS, SDP_PIGMENTS, build_ward_linkage, correlate_ratios, label_clusters, load_target_composites

EXPERIMENT = 'gulf_stream_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_analysis_config(EXPERIMENT)
identity_columns = ['polarity', 'track_id']
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
plankton_settings = read_settings(resolve_gold_dir(EXPERIMENT, 'eddy_plankton_table.parquet'), 'collocate-plankton')

eddy_tracks = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet'), cfg)
# Only the eddies first detected on one side of the axis and last detected on the other: cyclones from north to south, anticyclones from south to north
target_tracks = eddy_tracks.loc[eddy_tracks['movement'].eq(cast(pd.Series, eddy_tracks['polarity']).map(target_classes)), identity_columns]
target_keys = set(zip(target_tracks['polarity'], target_tracks['track_id']))
date_index = load_date_eddy_index(resolve_output_dir(EXPERIMENT, 'eddy_track'))
centerlines = index_centerlines_by_date(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet'))

pigments = list(SDP_PIGMENTS)
accessory = list(RATIO_PIGMENTS)
pigment_labels = {
    'Fuco': 'Fucoxanthin', 'HexFuco': "19'-Hex-fucoxanthin", 'ButFuco': "19'-But-fucoxanthin",
    'Perid': 'Peridinin', 'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin',
    'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
}

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    # matplotlib 3.11 warns about its default calligraphic face, Apple Chancery, even though no label uses it
    'mathtext.cal': 'Arial',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

## PACE Rrs against the SDP training range

SDP learned its pigment weights from 145 reflectance spectra measured in the water (Kramer et al. 2022). A PACE spectrum brighter or darker than every one of them asks SDP to extrapolate. The figure compares the water on each side of the Gulf Stream with the part of that training set closest to it.

- The two gray lines are the lowest and the highest training Rrs at each wavelength, on SDP's 1 nm grid, over the 33 training stations in the western North Atlantic (25 to 55 N, 85 to 30 W: the SABOR, NAAMES, and RemSensPOC cruises, March to November). The other 112 come from the eastern Atlantic, the Pacific, and the Mediterranean; Polarstern stations in the eastern Atlantic reach red Rrs near 10$^{-6}$ sr$^{-1}$, which stretched the full range far below anything near the Gulf Stream. The training values are stored to 10$^{-6}$ sr$^{-1}$, so the lower line steps near 700 nm.
- North-side and south-side water is the 8-day PACE Rrs in the band 150 to 450 km from the daily axis, within 4 degrees of longitude of a target eddy, with the eddy and its 3 R surroundings cut out. Each target eddy-composite gives one mean spectrum per side, and each target eddy averages its composites, so every eddy counts once.
- Lines are the median over eddies, and the shading holds the middle 50% (25th to 75th percentile). PACE is drawn at the OCI band centers; OCI has no bands from 588 to 613 nm, so the lines are straight there.
- The table under the figure gives, for each side, the share of eddies whose mean spectrum leaves this western North Atlantic range at one band or more, and the bands where the median spectrum sits above its maximum. A spectrum outside this range can still lie inside the full 145-spectrum range, so the table measures distance from the nearby training waters rather than extrapolation by SDP as a whole.

In [ ]:
composites = list_composites(DATA_DIR / 'bronze/sst', date_index, target_keys)
rrs_files = {path.name.split('.')[1]: path for path in (DATA_DIR / 'bronze/pace_l3_8d').glob('*.nc')}
with xr.open_dataset(next(iter(rrs_files.values()))) as ds:
    wavelengths = ds['wavelength'].to_numpy()
    rrs_lon = ds['lon'].to_numpy()
    rrs_lat = ds['lat'].to_numpy()
rrs_lon_grid, rrs_lat_grid = np.meshgrid(rrs_lon, rrs_lat)
visible = np.flatnonzero((wavelengths >= 400) & (wavelengths <= 700))
side_rows = []
for path, midpoint, eddies, _ in composites:
    with xr.open_dataset(rrs_files[path.name.split('.')[1]]) as ds:
        field = ds['Rrs'].isel(wavelength=visible).to_numpy()  # (n_lat, n_lon, n_visible)
    centerline = centerlines[midpoint.date()]
    axis_km = compute_signed_distance_grid_km(centerline.lon, centerline.lat, rrs_lon, rrs_lat)
    for eddy in eddies:
        half_chord = (
            np.sin(np.radians(rrs_lat_grid - eddy.center_lat) / 2) ** 2
            + np.cos(np.radians(rrs_lat_grid)) * np.cos(np.radians(eddy.center_lat)) * np.sin(np.radians(rrs_lon_grid - eddy.center_lon) / 2) ** 2
        )
        radius_units = 2 * 6371 * np.arcsin(np.sqrt(half_chord)) / eddy.radius_km
        # Each side's water is the band 150 to 450 km from the axis within 4 degrees of longitude of the eddy, with the eddy and its 3 R surroundings cut out
        band = (np.abs(rrs_lon_grid - eddy.center_lon) <= 4) & (radius_units > 3)
        regions = {'north': band & (axis_km >= 150) & (axis_km <= 450), 'south': band & (axis_km <= -150) & (axis_km >= -450)}
        means = average_regions(field, regions, plankton_settings['min_pixels'], plankton_settings['min_coverage'])
        side_rows.extend({'polarity': eddy.polarity, 'track_id': eddy.track_id, 'date': midpoint, 'side': side, 'rrs': mean} for side, mean in means.items())
sides = restrict_to_analysis_period(pd.DataFrame(side_rows), cfg)
# average_regions returns an all-NaN spectrum when too few pixels on that side are clear
sides = sides.loc[[not np.isnan(rrs).all() for rrs in sides['rrs']]]
# One spectrum per eddy and side, the mean over its composites, so an eddy seen in many composites counts once
eddy_spectra = {str(side): np.stack([np.nanmean(np.stack(rows['rrs'].to_list()), axis=0) for _, rows in members.groupby(identity_columns)]) for side, members in sides.groupby('side')}

# The 145 in-water training spectra on SDP's 1 nm grid, 400 to 700 nm, and their stations, read from the 2026-10-13 SDP audit in the same row order
training_dir = PROJECT_ROOT / 'data/sdp_validation/gulf_stream_validity_20261013/envelope'
training_rrs = np.load(training_dir / 'train_d2.npz')['rrs']
training_stations = pd.read_parquet(training_dir / 'train_meta.parquet')
# Only the stations in the western North Atlantic (SABOR, NAAMES, RemSensPOC). Eastern Atlantic stations from the Polarstern transects reach red Rrs near 1e-6, far below any water near the Gulf Stream
nearby = training_stations['Latitude'].between(25, 55) & training_stations['Longitude'].between(-85, -30)
training_rrs = training_rrs[nearby.to_numpy()]
training_wavelengths = np.arange(400, 701)
training_low, training_high = training_rrs.min(axis=0), training_rrs.max(axis=0)
side_colors = {'north': '#2166ac', 'south': '#b2182b'}

range_fig, range_ax = cast(tuple[Figure, Axes], plt.subplots(figsize=(6.69, 3.6), layout='constrained'))
range_ax.plot(training_wavelengths, training_high, color='#555555', linewidth=1, label=f'SDP training range in the western North Atlantic, lowest and highest of {len(training_rrs)} spectra')
range_ax.plot(training_wavelengths, training_low, color='#555555', linewidth=1)
for side, spectra in eddy_spectra.items():
    q25, q50, q75 = np.nanpercentile(spectra, [25, 50, 75], axis=0)
    range_ax.fill_between(wavelengths[visible], q25, q75, color=side_colors[side], alpha=0.25, linewidth=0)
    range_ax.plot(wavelengths[visible], q50, color=side_colors[side], linewidth=1.5, label=f'{side.capitalize()}-side water, median and middle 50% (n = {len(spectra)} eddies)')
range_ax.set_yscale('log')
range_ax.set_xlim(400, 700)
range_ax.set_xlabel('Wavelength (nm)')
range_ax.set_ylabel('Rrs (sr$^{-1}$)')
range_ax.legend(loc='lower left')
plt.show()

# Training bounds at the OCI band centers, to count the eddies whose mean spectrum falls outside them
low_at_bands, high_at_bands = (np.interp(wavelengths[visible], training_wavelengths, bound) for bound in (training_low, training_high))
display(pd.DataFrame([
    {
        'side': side, 'eddies': len(spectra),
        'above the training maximum': np.mean((spectra > high_at_bands).any(axis=1)), 'below the training minimum': np.mean((spectra < low_at_bands).any(axis=1)),
        'bands where the median is above the maximum (nm)': ', '.join(f'{wavelength:.0f}' for wavelength in wavelengths[visible][np.median(spectra, axis=0) > high_at_bands]),
    }
    for side, spectra in eddy_spectra.items()
]).set_index('side').round(2))

## Pigment clustering

Based on the pigment concentration we observe in eddies, we want to determine how many distinct groups of PFTs we can tell apart via hierarchical clustering. That way, we may make a claim about abundance for a group of PFTs.

We perform this analysis for our target eddies only. Each sample is one eddy-composite: the pigments are first averaged over all interior pixels of a valid eddy-composite, and that mean is one point in the correlations. We average first rather than treating every pixel as a sample, since single-pixel SDP retrievals are much noisier.


1. We divide each of the pigments by Tchla (13 total, with 1 being Tchla, so we have 12 accessory pigments). That way, if two eddies contain the same community composition in terms of PFTs they will have the same pigment ratios regardless of different total biomass.
    - This is NOT normalized.

2. Compute a matrix of Pearson's correlation between every pair of pigment ratios
    - We use 8-day PACE composites. If an eddy is spotted multiple times within that 8-day period it will only count once (otherwise we'd be duplicating the same PACE observations), choosing the contour on a day nearest to the center of the 8-day composite.

3. Use $\sqrt{2(1 - r)}$ as distance metric between two points (0-2 range inclusive). Build with Ward's linkage combination strategy for hierarchical clustering.

4. After determining a cutoff for clusters of pigments, we have groups of pigments that need to be mapped to groups of PFTs. Following Kramer and Siegel (2019), a cluster is named after every PFT whose diagnostic (marker) pigment it contains. Pigments shared by several PFTs (Chlc12, Chlc3, Viola) don't name a cluster; they just sit with whichever markers they correlate with.

Chlorophyll-a is the one pigment that every photosynthetic phytoplankton group carries. However, it's not a direct proxy for biomass since different PFTs can have different amounts of Chl-a per unit of carbon/biomass.

### Target eddies and their composites

In [ ]:
# One row per target eddy-composite, with the interior-mean pigments: cyclones that crossed the axis north to south, anticyclones south to north
target_pigments = load_target_composites(DATA_DIR, cfg)

### Sample counts

Checkpoint: the number of target eddies, eddy-composites per polarity.

In [ ]:
display(cast(pd.DataFrame, target_pigments.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
)))

### Correlation matrix

The 12 x 12 correlation matrix of the pigment-to-Tchla ratios.

In [ ]:
# Each eddy-composite is one data point: every pigment is the mean over the composite's interior pixels, divided by the interior-mean Tchla
correlation = correlate_ratios(target_pigments)

### Cluster the pigments and draw the tree

Every sample comes from inside a target eddy. No ring around the eddy and no background water enter, so the tree shows which pigment ratios rise and fall together from one target eddy-composite to the next. It says nothing about whether the water outside the eddies holds the same groups.

- Target eddies are cyclones that crossed the Gulf Stream axis from north to south and anticyclones that crossed it from south to north, within `analysis_date_range`.
- Only PACE pixels inside the eddy's speed contour count, using the contour from the day nearest the middle of the 8-day composite. A composite enters only when at least 80% of those interior pixels hold a valid Rrs spectrum and there are at least 10 of them.
- Each pigment is the mean SDP value over the interior pixels, clipped at zero after averaging so that negative and positive pixel noise cancel first. The ratio divides that mean by the interior-mean Tchla of the same composite.
- `keep_valid_sdp_rows` drops a composite when Tchla is zero or below on half or more of its interior pixels, when the mean DV_chla is at or above the mean Tchla (DV_chla is part of Tchla), or when the 11 accessory pigments sum to more than 1.64 times Tchla, the largest ratio among the 145 samples SDP was trained on. It removes about a third of all eddy-composites before the target eddies are picked.
- Each eddy-composite is one sample, so an eddy seen on more composites weighs more in the correlation. The resampling under "Choosing the cut height" draws whole eddies to test how much the splits depend on which eddies are in the sample.

Reading the tree:

- The dashed line is `CLUSTER_CUT_HEIGHT`. Pigments joined below it share a color, and a pigment alone in its cluster stays gray.
- The bracket under each cluster names the PFTs whose diagnostic pigments it holds, following the map below the tree. Chlorophyll-c3 is the diagnostic pigment of no PFT, so it has no bracket.
- The tree's cophenetic r, the correlation between its merge heights and the original distances between pigments, is printed under the tree; the closer to 1, the better the tree keeps those distances. Kramer and Siegel (2019) check it before cutting.

In [ ]:
def draw_tree(corr: np.ndarray, cutoff: float) -> np.ndarray:
    set_link_color_palette(['#1b9e77', '#d95f02', '#7570b3', '#e7298a'])
    distance, Z = build_ward_linkage(corr)
    cluster_numbers = fcluster(Z, t=cutoff, criterion='distance')
    cluster_fig, tree_ax = cast('tuple[Figure, Axes]', plt.subplots(figsize=(6.69, 4.6)))
    cluster_fig.subplots_adjust(left=0.075, right=0.99, bottom=0.39, top=0.98)
    tree = dendrogram(Z, ax=tree_ax, labels=[pigment_labels[pigment] for pigment in accessory], leaf_rotation=90, leaf_font_size=8, color_threshold=cutoff, above_threshold_color='#444444')
    for label, color in zip(tree_ax.xaxis.get_ticklabels(), tree['leaves_color_list']):
        label.set_color(color)
    tree_ax.axhline(cutoff, color='#999999', linewidth=0.6, linestyle=(0, (4, 2.5)))  # pyright: ignore[reportArgumentType]
    tree_ax.tick_params(axis='x', length=0)
    tree_ax.autoscale(axis='y')
    tree_ax.set_ylim(0, tree_ax.get_ylim()[1])
    tree_ax.yaxis.set_major_locator(MaxNLocator(nbins=8, steps=[1, 2, 5, 10]))
    tree_ax.yaxis.grid(True, color='#e8e8e8', linewidth=0.5)
    tree_ax.set_axisbelow(True)
    for spine in tree_ax.spines.values():
        spine.set_visible(True)
    tree_ax.set_ylabel('Linkage distance')
    # Under the leaf names of each cluster, a bracket and the groups whose diagnostic pigments sit in it, in figure coordinates below the lowest leaf name
    cluster_labels = {cluster.cluster_id: cluster.labels for cluster in label_clusters(pd.Series(cluster_numbers, index=accessory))}
    cluster_fig.draw_without_rendering()
    renderer = cluster_fig.canvas.get_renderer()  # pyright: ignore[reportAttributeAccessIssue]
    leaf_bottom = cluster_fig.transFigure.inverted().transform((0, min(label.get_window_extent(renderer).y0 for label in tree_ax.get_xticklabels())))[1]
    point = 1 / 72 / cluster_fig.get_figheight()
    bracket_top, bracket_bottom = leaf_bottom - 4 * point, leaf_bottom - 8 * point
    below_axis = blended_transform_factory(tree_ax.transData, cluster_fig.transFigure)
    group_labels = []
    for cluster_id in np.unique(cluster_numbers):
        if not cluster_labels[cluster_id]:
            continue
        members = [position for position, leaf in enumerate(tree['leaves']) if cluster_numbers[leaf] == cluster_id]
        # By cluster number, not leaf color: every single-pigment cluster takes the same above-cutoff gray. Leaves sit at x = 5, 15, 25, ...
        left, right = 10 * min(members) + 1, 10 * max(members) + 9
        middle = (left + right) / 2
        color = tree['leaves_color_list'][members[0]]
        cluster_fig.add_artist(Line2D([left, left, right, right], [bracket_top, bracket_bottom, bracket_bottom, bracket_top], transform=below_axis, color=color, linewidth=0.8))
        cluster_fig.add_artist(Line2D([middle, middle], [bracket_bottom, bracket_bottom - 3 * point], transform=below_axis, color=color, linewidth=0.8))
        group_labels.append(tree_ax.text(middle, bracket_bottom - 5 * point, ',\n'.join(cluster_labels[cluster_id]), transform=below_axis, ha='center', va='top', color=color))
    # Keep every label on one line: widen the figure in 5% steps until each label clears its neighbors by a few points
    for _ in range(40):
        boxes = sorted((label.get_window_extent(renderer) for label in group_labels), key=lambda box: box.x0)
        if all(right.x0 >= left.x1 + 0.1 * cluster_fig.dpi for left, right in zip(boxes, boxes[1:])):
            break
        cluster_fig.set_size_inches(1.05 * cluster_fig.get_figwidth(), cluster_fig.get_figheight())
    plt.show()
    # The correlation between the tree's merge heights and the distances it was built from, Kramer and Siegel's check before cutting
    cophenetic_r, _ = cophenet(Z, distance)
    print(f'cophenetic r: {cophenetic_r:.3f}')
    return cluster_numbers


# CLUSTER_CUT_HEIGHT sits inside the gap between the merges at 0.68 and 0.76 of the ratio tree, the six-cluster level
cluster_numbers = draw_tree(correlation.to_numpy(), cutoff=CLUSTER_CUT_HEIGHT)

#### Pigment-to-group map

The pigments each group carries, from Table 1 of Kramer and Siegel (2019). Bold marks the diagnostic pigments, the only ones that name a cluster under the tree, as in Vidussi et al. (2001) and Uitz et al. (2006). Chlc12, Chlc3, and Viola occur in several groups and never name one.

- Diatoms: **Fuco**, Chlc12, Chlc3
- Dinoflagellates: Fuco, **Perid**, Chlc12, Chlc3
- Chrysophytes: Fuco, Chlc12, Viola, Zea (no diagnostic pigment among the SDP pigments, so no cluster carries their name)
- Pelagophytes: Fuco, **ButFuco**, Chlc12, Zea
- Haptophytes: Fuco, **HexFuco**, ButFuco, Chlc12, Chlc3
- Cryptophytes: Chlc12, **Allo**
- Green algae: **MV_chlb**, **Neo**, Viola, Zea
- Cyanobacteria: **Zea**, **DV_chla**

The map, the cut height, and the labeling rule live in `eddy_tracking.utils.pigment_clusters`, which `figures.ipynb` also reads.

#### Choosing the cut height

Kramer and Siegel (2019) cut each tree at the linkage distance that gives "a maximum number of taxonomically relevant clusters." Read from the top of the tree with the diagnostic-pigment labels:

- 3 clusters (cut 1.45 to 2.60): diatoms; dinoflagellates, pelagophytes, haptophytes, and cyanobacteria; cryptophytes and green algae.
- 4 clusters (0.99 to 1.45): Chlc3 splits off alone. It carries no marker, so no new group.
- 5 clusters (0.76 to 0.99): Perid splits off, giving dinoflagellates their own cluster.
- 6 clusters (0.68 to 0.76): Zea and DV_chla split from HexFuco and ButFuco, separating cyanobacteria from haptophytes and pelagophytes.
- 7 clusters and below: Neo splits from the green algae, then Fuco from Chlc12. These repeat groups that already have a cluster.

The six-cluster level is the lowest cut where every split still separates a new group, so `CLUSTER_CUT_HEIGHT` is 0.72. Its window (0.68 to 0.76) is much narrower than the three-cluster window (1.45 to 2.60), so the cell below resamples whole eddies to check how often the two key splits survive. The three-cluster cut serves as the robustness check.

In [ ]:
# Resample whole eddies with replacement, rebuild the ratio tree, cut it into the same number of clusters, and count how often each split holds
N_RESAMPLES = 500
rng = np.random.default_rng(2026)
eddy_keys = target_pigments[['polarity', 'track_id']].drop_duplicates()
by_eddy = {key: rows for key, rows in target_pigments.groupby(['polarity', 'track_id'])}
n_main = len(np.unique(cluster_numbers))
split_rows = []
for n_clusters in (3, n_main):
    agreement = []
    cyanobacteria_apart = perid_alone = 0
    full = fcluster(build_ward_linkage(correlation.to_numpy())[1], t=n_clusters, criterion='maxclust')
    upper = np.triu_indices(len(accessory), 1)
    for _ in range(N_RESAMPLES):
        sample = pd.concat([by_eddy[tuple(key)] for key in eddy_keys.to_numpy()[rng.integers(0, len(eddy_keys), len(eddy_keys))]])
        resampled = fcluster(build_ward_linkage(correlate_ratios(sample).to_numpy())[1], t=n_clusters, criterion='maxclust')
        # Share of pigment pairs that are together (or apart) in both the resampled and the full tree
        agreement.append(np.mean((full[:, None] == full[None, :])[upper] == (resampled[:, None] == resampled[None, :])[upper]))
        cluster_of = dict(zip(accessory, resampled))
        cyanobacteria_apart += cluster_of['Zea'] == cluster_of['DV_chla'] and cluster_of['HexFuco'] == cluster_of['ButFuco'] and cluster_of['Zea'] != cluster_of['HexFuco']
        perid_alone += cluster_of['Perid'] not in (cluster_of['HexFuco'], cluster_of['Zea'], cluster_of['Fuco'])
    split_rows.append({
        'clusters': n_clusters, 'median pair agreement': np.median(agreement), '5th percentile': np.percentile(agreement, 5),
        'cyanobacteria apart from haptophytes': cyanobacteria_apart / N_RESAMPLES, 'Perid on its own': perid_alone / N_RESAMPLES,
    })
display(pd.DataFrame(split_rows).round(2))

The grouping of alloxanthin with monovinyl chlorophyll-b, neoxanthin, and violaxanthin matches observations of correlated pigment-to-total-chlorophyll-a ratios in the western North Atlantic ([Kramer et al., 2020](https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2020.00215/full)). This association also appears globally, although its strength varies by region ([Kramer & Siegel, 2019](https://pmc.ncbi.nlm.nih.gov/articles/PMC7043335/)). It supports cryptophyte and green-algal pigment covariation, which is why this cluster carries both labels; it does not establish a universal correlation between their abundances.